# 01 Data Preprocessing
This notebook demonstrates the Phase 1 Data Collection & Preprocessing pipeline.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data.downloader import main as download_main

# Cell 2: Run downloader (or skip if data exists)
download_main()

In [ ]:
from src.data.loader import load_dataset, get_feature_matrix

# Cell 3: Load raw dataset, display .info(), .describe(), .head()
df = load_dataset()
display(df.info())
display(df.describe())
display(df.head())

In [ ]:
from src.data.cleaner import clean_pipeline

# Cell 4: Run cleaner pipeline, show before/after statistics
df_clean = clean_pipeline(df)
display(df_clean.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import FEATURE_COLUMNS

# Cell 5: Visualize data quality heatmap (nulls per feature)
plt.figure(figsize=(12, 6))
sns.heatmap(df[[c for c in FEATURE_COLUMNS if c in df.columns]].isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap (Before Cleaning)')
plt.show()

In [ ]:
from src.data.splitter import stratified_split, apply_smote, save_splits

# Cell 6: Run splitter, show class distribution bar charts
train, val, test = stratified_split(df_clean)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
label_col = 'Class' if 'Class' in df_clean.columns else df_clean.columns[-1]
sns.countplot(x=label_col, data=train, ax=axes[0]).set_title('Train Class Distribution')
sns.countplot(x=label_col, data=val, ax=axes[1]).set_title('Val Class Distribution')
sns.countplot(x=label_col, data=test, ax=axes[2]).set_title('Test Class Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Show SMOTE before/after comparison
train_balanced = apply_smote(train)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
sns.countplot(x=label_col, data=train, ax=axes[0]).set_title('Before SMOTE')
sns.countplot(x=label_col, data=train_balanced, ax=axes[1]).set_title('After SMOTE')
plt.tight_layout()
plt.show()

# Finally, save the splits
save_splits(train_balanced, val, test)

# Summary
- Downloaded and validated dataset.
- Loaded dataset from CSV.
- Cleaned data: removed duplicates, handled missing values, removed constants, detected outliers.
- Split data into train/val/test using stratified splitting.
- Handled class imbalance in the training data using SMOTE.
- Saved optimized subsets to Disk.